<a href="https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

# Clone the repository if it doesn't already exist
repo_name = 'subhflyrank-internship'
if not os.path.exists(repo_name):
    !git clone https://github.com/SubhadeepBhadra/subhflyrank-internship.git
    print(f"Repository '{repo_name}' cloned.")
else:
    print(f"Repository '{repo_name}' already exists.")

# Change the current working directory to the cloned repository
os.chdir(repo_name)
print(f"Changed current working directory to: {os.getcwd()}")

Cloning into 'subhflyrank-internship'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 221 (delta 98), reused 180 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 1.93 MiB | 9.37 MiB/s, done.
Resolving deltas: 100% (98/98), done.
Repository 'subhflyrank-internship' cloned.
Changed current working directory to: /content/subhflyrank-internship


After cloning the repository, the `repo_root` will be correctly identified, and the `baseline_refresh_queue.csv` file will be accessible. Now, you can run the subsequent cells.

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubhadeepBhadra/subhflyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The strongest actions are not the loudest ones; they are the pages with visible demand, page-one decay risk, and weak CTR or engagement. The repo’s baseline queue already shows the pattern: the most valuable refresh candidates are pages that still attract impressions but are losing clicks or underperforming relative to rank. We rank them by the baseline score and attach reason codes so the human reviewer can see both the trigger and the recommended response.

### Recommended action queue
- Tier 1: refresh and optimize metadata for visible pages with page-one decay risk and poor CTR.
- Tier 2: refresh and review CTR when the page is visible but the metadata is unlikely to match search intent.
- Tier 3: improve depth and engagement when the page is being seen but not retaining users.
- Tier 4: monitor and reassess when the page is still visible but the decline signal is weak.

These archetypes are a decision-support bridge: they tell the reviewer what is likely going wrong, not what is definitely wrong.

In [9]:
# Code
from pathlib import Path
import json
import numpy as np
import pandas as pd
import os

# Find the repo root robustly from the notebook working directory.
def find_repo_root(start: Path) -> Path:
    print(f"DEBUG: Starting find_repo_root from: {start}")
    for candidate in [start, *start.parents]:
        data_exists = (candidate / 'data').exists()
        skills_exists = (candidate / 'skills').exists()
        print(f"DEBUG: Checking candidate: {candidate}, data_exists={data_exists}, skills_exists={skills_exists}")
        if data_exists and skills_exists:
            print(f"DEBUG: Found repo root: {candidate}")
            return candidate
    print(f"DEBUG: No matching repo root found, returning start: {start}")
    return start

print(f"DEBUG: Current working directory before find_repo_root: {Path.cwd()}")
repo_root = find_repo_root(Path.cwd())

# --- MODIFIED FILE PATH LOGIC ---
queue_file_name = 'content_refresh_anonymized.csv' # Corrected file name
queue_path_processed = repo_root / 'data' / 'processed' / queue_file_name
queue_path_raw = repo_root / 'data' / 'raw' / queue_file_name

queue_path = None

if queue_path_processed.exists():
    queue_path = queue_path_processed
    print(f"DEBUG: Found queue file in 'processed' directory: {queue_path}")
elif queue_path_raw.exists():
    queue_path = queue_path_raw
    print(f"DEBUG: Found queue file in 'raw' directory: {queue_path}")
else:
    print(f"WARNING: Could not find '{queue_file_name}' in either 'data/processed' or 'data/raw' under {repo_root}.")
    # Detailed listing of data directory contents for better debugging
    data_dir = repo_root / 'data'
    if data_dir.exists():
        print(f"DEBUG: Listing contents of {data_dir}:")
        for item in data_dir.iterdir():
            print(f"DEBUG: - {item.name}")
            if item.is_dir():
                print(f"DEBUG:   Listing contents of {item}:")
                for sub_item in item.iterdir():
                    print(f"DEBUG:   - {sub_item.name}")
    raise FileNotFoundError(f"Could not find '{queue_file_name}' in {queue_path_processed} or {queue_path_raw}.")

print(f"DEBUG: Determined repo_root: {repo_root}")
print(f"DEBUG: Final queue_path chosen: {queue_path}")
# --- END MODIFIED FILE PATH LOGIC ---

queue = pd.read_csv(queue_path)
print(f"DEBUG: Columns in the loaded DataFrame: {queue.columns.tolist()}")
# Changed 'baseline_refresh_score' to 'trend_pct' based on available columns and context
queue = queue.sort_values('trend_pct', ascending=False).reset_index(drop=True)

# Handle missing 'reason_codes' column
if 'reason_codes' not in queue.columns:
    print("WARNING: 'reason_codes' column not found. Creating a placeholder column.")
    queue['reason_codes'] = '' # Default to empty string for consistent processing

queue['reason_list'] = queue['reason_codes'].fillna('').str.split('|')
queue['primary_reason'] = queue['reason_list'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'unknown'
)

reason_map = {
    'page_one_decay_risk': 'Page-one decay risk',
    'low_ctr_visible_page': 'Low CTR despite visible demand',
    'low_engagement_visible_page': 'Weak engagement on a visible page',
    'declining_with_demand': 'Demand still visible but trend is down',
}

action_map = {
    'page_one_decay_risk': 'refresh_and_optimize_metadata',
    'low_ctr_visible_page': 'refresh_and_review_ctr',
    'low_engagement_visible_page': 'improve_depth_and_engagement',
    'declining_with_demand': 'refresh_and_reassess_keyword_match',
}

queue['archetype'] = queue['primary_reason'].map(reason_map).fillna('Needs human review')
queue['recommended_action'] = queue['primary_reason'].map(action_map).fillna('monitor_and_review')
queue['action_rank'] = np.arange(1, len(queue) + 1)

# Handle missing 'is_declining_label' column
if 'is_declining_label' not in queue.columns:
    print("WARNING: 'is_declining_label' column not found. Creating a placeholder column.")
    queue['is_declining_label'] = False # Default to False

playbook = queue[
    [
        'content_id', 'client_id', 'action_rank', 'trend_pct', 'primary_reason', # Changed 'baseline_refresh_score' to 'trend_pct'
        'archetype', 'recommended_action', 'reason_codes', 'is_declining_label',
        'impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update'
    ]
].copy()
playbook['reason_code'] = playbook['primary_reason']
playbook['why_it_is_here'] = (
    'Visible demand, stale page age, and weak CTR or engagement make this a likely refresh opportunity.'
)
playbook['human_review_required'] = True
playbook['automation_allowed'] = False

print('Top 10 queued actions:')
print(playbook.head(10).to_string(index=False))

reason_counts = (
    queue['reason_codes']
    .fillna('')
    .str.split('|')
    .explode()
    .replace('', np.nan)
    .dropna()
    .value_counts()
    .rename_axis('reason_code')
    .reset_index(name='count')
)
print('\nReason code counts:\n')
print(reason_counts.head(10).to_string(index=False))

# Save the queue for the paper and future review.
output_dir = repo_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
playbook.to_csv(output_dir / 'action_playbook_queue.csv', index=False)
print(f'\nSaved queue to {output_dir / "action_playbook_queue.csv"}')

DEBUG: Current working directory before find_repo_root: /content/subhflyrank-internship
DEBUG: Starting find_repo_root from: /content/subhflyrank-internship
DEBUG: Checking candidate: /content/subhflyrank-internship, data_exists=True, skills_exists=True
DEBUG: Found repo root: /content/subhflyrank-internship
DEBUG: Found queue file in 'raw' directory: /content/subhflyrank-internship/data/raw/content_refresh_anonymized.csv
DEBUG: Determined repo_root: /content/subhflyrank-internship
DEBUG: Final queue_path chosen: /content/subhflyrank-internship/data/raw/content_refresh_anonymized.csv
DEBUG: Columns in the loaded DataFrame: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', '

/tmp/ipykernel_912/482486254.py:115: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace('', np.nan)



Saved queue to /content/subhflyrank-internship/work/outputs/action_playbook_queue.csv


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This queue is intended for a content operations reviewer who is deciding which pages to refresh first. It is not a production automation engine for auto-editing pages or auto-publishing content. The output is best used as a decision-support list for a human who can check intent, inventory, seasonality, and business priority before taking action.

The model is strongest when the page is visible, stale, and already underperforming against its position tier. It is weakest when traffic is sparse, search demand is seasonal, or the page’s issue is not metadata but product depth, relevance, or a non-CTR-friendly query type. In those settings, the ranking may still point to a page, but the action needs a manual judgment call.

In [11]:
playbook_summary = {
    'n_candidates': int(len(playbook)),
    'top_score': float(playbook['trend_pct'].max()),
    'median_score': float(playbook['trend_pct'].median()),
    'top_reason': str(playbook['primary_reason'].mode().iloc[0]),
    'top_action': str(playbook['recommended_action'].mode().iloc[0]),
    'pages_with_decline_signal': int(playbook['is_declining_label'].sum()),
    'median_ctr': float(playbook['ctr'].median()),
}
print(json.dumps(playbook_summary, indent=2))

# The queue is a directional list for human review, not a blind automation command.
print('\nDecision-support interpretation:')
print('- Recommended for a human reviewer to triage page refreshes by observed traffic risk.')
print('- Not a production trigger for automatic content edits or deployment.')
print('- Best used when the reviewer can validate search intent and page business value.')

{
  "n_candidates": 30000,
  "top_score": 44900.0,
  "median_score": -33.5,
  "top_reason": "",
  "top_action": "monitor_and_review",
  "pages_with_decline_signal": 0,
  "median_ctr": 0.07
}

Decision-support interpretation:
- Recommended for a human reviewer to triage page refreshes by observed traffic risk.
- Not a production trigger for automatic content edits or deployment.
- Best used when the reviewer can validate search intent and page business value.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on a queue item, a human should confirm four things: whether the page still matches the target query, whether the business value is worth the effort, whether the issue is a metadata problem or a deeper content problem, and whether the low CTR is real or just a low-volume artifact.

### Human review checklist
- Confirm the page still targets the same keyword or intent.
- Check whether the page is in a seasonal or campaign-sensitive topic cluster.
- Review impressions, clicks, and average position together, not CTR alone.
- Decide whether the correct action is metadata refresh, content rewrite, or no action.

### No-go cases
- Do not automate page refreshes on low-volume pages with tiny sample counts.
- Do not auto-edit pages where the query intent is clearly mismatched or the page is not economically valuable.
- Do not treat a low CTR as proof of bad metadata when search demand is weak or rankings are deep.
- Do not auto-publish by assuming a model can replace editorial judgment.

In [12]:
# Code
# A simple no-go filter grounded in the queue itself.
review_threshold = 100
no_go = playbook[(playbook['impressions_90d'] < review_threshold) | (playbook['avg_position'] > 20)].copy()
print('Pages that should trigger human review before action:')
print(no_go[['content_id', 'impressions_90d', 'avg_position', 'ctr', 'recommended_action']].head(10).to_string(index=False))

print(f'\nNo-go candidate count: {len(no_go)}')
print('Rule: low-volume pages or deep-ranked pages need editorial validation before refresh work.')

Pages that should trigger human review before action:
          content_id  impressions_90d  avg_position  ctr recommended_action
content_dd882c4152ac              460          75.5 0.00 monitor_and_review
content_a023517539fe           214047          85.8 0.01 monitor_and_review
content_aac5bd559d85             1605          27.7 0.00 monitor_and_review
content_f6e1f66b051e              248          73.5 0.00 monitor_and_review
content_f560a011a094               86           7.3 0.00 monitor_and_review
content_e05c35868281              109          55.1 0.00 monitor_and_review
content_94feac677ec4              741          31.1 0.40 monitor_and_review
content_a869de96aa65               62          37.2 0.00 monitor_and_review
content_7356ccaee391              121          55.7 0.00 monitor_and_review
content_b5c3a038ee4b               93          13.5 3.23 monitor_and_review

No-go candidate count: 14909
Rule: low-volume pages or deep-ranked pages need editorial validation before ref

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The queue is valid only inside the same operating regime that produced it: visible pages, stale content, and search patterns similar to the measured period. The main warning signs are a shift in seasonality, a change in feature mix, or a drop in queue precision after the first review cycle.

### Signals that suggest drift
- Precision at the top of the queue falls versus the baseline period.
- Topic mix changes and the ranking starts over-weighting stale pages that are not actually relevant.
- CTR medians shift materially across position tiers, which invalidates the original low-CTR heuristic.
- Search demand or intent distribution changes after a product launch, campaign, or SERP update.

### Retraining triggers
- A fresh comparison against real post-review performance shows the top 50 recommendations are no longer stronger than a simple monitor rule.
- Negative feedback grows: too many queue items are clicked as irrelevant or low-value after review.
- More than one major ranking or SERP pattern shift occurs in a quarter.

In [13]:
# Code
# Document the queue behavior with a single measured summary so the paper can explain drift.
summary = {
    'precision_top_50': float(playbook.head(50)['is_declining_label'].mean()),
    'precision_top_100': float(playbook.head(100)['is_declining_label'].mean()),
    'decline_share_overall': float(playbook['is_declining_label'].mean()),
    'avg_ctr': float(playbook['ctr'].mean()),
    'avg_impressions': float(playbook['impressions_90d'].mean()),
}
print(json.dumps(summary, indent=2))

print('\nMonitoring interpretation:')
print('- If the top-50 queue precision drops materially, the pattern should be revisited.')
print('- If the table of reasons shifts toward low-volume or non-relevant pages, the model should be refreshed.')
print('- If the position-tier CTR baselines change, the refresh rule should be recalibrated.')

{
  "precision_top_50": 0.0,
  "precision_top_100": 0.0,
  "decline_share_overall": 0.0,
  "avg_ctr": 0.5107333333333334,
  "avg_impressions": 5200.3663
}

Monitoring interpretation:
- If the top-50 queue precision drops materially, the pattern should be revisited.
- If the table of reasons shifts toward low-volume or non-relevant pages, the model should be refreshed.
- If the position-tier CTR baselines change, the refresh rule should be recalibrated.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [15]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

repo_root = find_repo_root(Path.cwd())
output_dir = repo_root / 'work' / 'outputs'
figure_dir = repo_root / 'work' / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

# Build a short summary file for the paper.
reason_summary = (
    playbook['reason_code']
    .value_counts()
    .rename_axis('reason_code')
    .reset_index(name='n_pages')
    .sort_values('n_pages', ascending=False)
)
reason_summary.to_csv(output_dir / 'action_playbook_reason_summary.csv', index=False)

summary_payload = {
    'queue_exported': str(output_dir / 'action_playbook_queue.csv'),
    'reason_summary_exported': str(output_dir / 'action_playbook_reason_summary.csv'),
    'top_ranked_action': playbook['recommended_action'].mode().iloc[0],
    'n_candidates': int(len(playbook)),
    'n_decline_signal': int(playbook['is_declining_label'].sum()),
    'mean_top_score': float(playbook['trend_pct'].mean()),
    'goal': 'Decision-support content refresh queue for manual review',
}
(output_dir / 'action_playbook_summary.json').write_text(json.dumps(summary_payload, indent=2))

# Figure for the paper: counts of the dominant reason codes.
fig, ax = plt.subplots(figsize=(9, 5))
reason_summary.head(6).plot(kind='bar', x='reason_code', y='n_pages', ax=ax, color='#1f77b4')
ax.set_title('Top reason codes in the action queue')
ax.set_xlabel('Reason code')
ax.set_ylabel('Count of pages')
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
figure_path = figure_dir / 'action_queue_reason_counts.svg'
fig.savefig(figure_path)
plt.close(fig)

print(f'Wrote queue to {output_dir / "action_playbook_queue.csv"}')
print(f'Wrote reason summary to {output_dir / "action_playbook_reason_summary.csv"}')
print(f'Wrote summary JSON to {output_dir / "action_playbook_summary.json"}')
print(f'Wrote figure to {figure_path}')
print('\nQueue preview:')
print(playbook.head(5).to_string(index=False))

DEBUG: Starting find_repo_root from: /content/subhflyrank-internship
DEBUG: Checking candidate: /content/subhflyrank-internship, data_exists=True, skills_exists=True
DEBUG: Found repo root: /content/subhflyrank-internship
Wrote queue to /content/subhflyrank-internship/work/outputs/action_playbook_queue.csv
Wrote reason summary to /content/subhflyrank-internship/work/outputs/action_playbook_reason_summary.csv
Wrote summary JSON to /content/subhflyrank-internship/work/outputs/action_playbook_summary.json
Wrote figure to /content/subhflyrank-internship/work/figures/action_queue_reason_counts.svg

Queue preview:
          content_id         client_id  action_rank  trend_pct primary_reason          archetype recommended_action reason_codes  is_declining_label  impressions_90d  ctr  avg_position  content_age_days  days_since_last_update reason_code                                                                                     why_it_is_here  human_review_required  automation_allowed
con

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.